In [ ]:
%pip install python-dotenv

In [ ]:
import os
import base64
from openai import AzureOpenAI
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()

In [ ]:
#이건 OpenAI 코드
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_API_KEY")

# 키 기반 인증을 사용하여 Azure OpenAI 클라이언트 초기화
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)

# IMAGE_PATH = "YOUR_IMAGE_PATH"
# encoded_image = base64.b64encode(open(IMAGE_PATH, 'rb').read()).decode('ascii')

# 채팅 프롬프트 준비
chat_prompt = [
    {
        "role": "system",
        "content": "사용자가 정보를 찾는 데 도움이 되는 AI 도우미입니다."
    },
    {
        "role": "user",
        "content": "안녕하세요. 도움이 필요합니다."
    }
]

# 음성이 사용되는 경우 음성 결과 포함
messages = chat_prompt

# 완료 생성
completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    max_tokens=800,
    temperature=1,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
    stream=False,
    extra_body={
      "data_sources": [{
          "type": "azure_search",
          "parameters": {
            "endpoint": search_endpoint,
            "index_name": "team-5-perror",
            "semantic_configuration": "team-5-perror-semantic-configuration",
            "query_type": "vector_semantic_hybrid",
            "fields_mapping": {
              "content_fields_separator": "\n",
              "content_fields": None,
              "filepath_field": None,
              "title_field": "title",
              "url_field": None,
              "vector_fields": [
                "text_vector"
              ]
            },
            "in_scope": True,
            "filter": None,
            "strictness": 3,
            "top_n_documents": 5,
            "authentication": {
              "type": "api_key",
              "key": search_key
            },
            "embedding_dependency": {
              "type": "deployment_name",
              "deployment_name": "text-embedding-3-large"
            }
          }
        }]
    }
)

print(completion.to_json())

In [ ]:
#이건 TTS 서비스 코드
import os
# from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

# load_dotenv()

speech_key = os.getenv("AZURE_SPEECH_KEY")
speech_region = os.getenv("AZURE_SPEECH_REGION")

speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)
# Note: the voice setting will not overwrite the voice element in input SSML.
speech_config.speech_synthesis_voice_name = "ko-KR-HyunsuMultilingualNeural"

text = "Hi, this is Hyunsu Multilingual"

# use the default speaker as audio output.
speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config)

result = speech_synthesizer.speak_text_async(text).get()
# Check result
if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print("Speech synthesized for text [{}]".format(text))
elif result.reason == speechsdk.ResultReason.Canceled:
    cancellation_details = result.cancellation_details
    print("Speech synthesis canceled: {}".format(cancellation_details.reason))
    if cancellation_details.reason == speechsdk.CancellationReason.Error:
        print("Error details: {}".format(cancellation_details.error_details))

In [ ]:
#이건 Openai + tts 코드 묶은 함수로 것
import os
import base64
import json
from openai import AzureOpenAI
import azure.cognitiveservices.speech as speechsdk
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv(override=True)

def azure_openai_with_speech(user_question, voice_name="ko-KR-HyunsuMultilingualNeural"):
    # 환경변수에서 설정값 불러오기
    endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
    deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

    search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
    search_key = os.getenv("AZURE_SEARCH_API_KEY")
    search_index = os.getenv("SEARCH_INDEX")  # 소문자로 되어 있으면 여기도 일치시켜야 함

    speech_key = os.getenv("AZURE_SPEECH_KEY")
    service_region = os.getenv("AZURE_SPEECH_REGION")

    # 키 기반 인증을 사용하여 Azure OpenAI 클라이언트 초기화
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=subscription_key,
        api_version="2025-01-01-preview",
    )

    # 채팅 프롬프트 준비
    chat_prompt = [
        {
            "role": "system",
            "content": "사용자가 정보를 찾는 데 도움이 되는 AI 도우미입니다."
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    # 완료 생성
    completion = client.chat.completions.create(
        model=deployment,
        messages=chat_prompt,
        max_tokens=800,
        temperature=1,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None,
        stream=False,
        extra_body={
            "data_sources": [{
                "type": "azure_search",
                "parameters": {
                    "endpoint": search_endpoint,
                    "index_name": search_index,
                    "semantic_configuration": f"{search_index}-semantic-configuration",
                    "query_type": "vector_semantic_hybrid",
                    "fields_mapping": {
                        "content_fields_separator": "\n",
                        "content_fields": None,
                        "filepath_field": None,
                        "title_field": "title",
                        "url_field": None,
                        "vector_fields": ["text_vector"]
                    },
                    "in_scope": True,
                    "filter": None,
                    "strictness": 3,
                    "top_n_documents": 5,
                    "authentication": {
                        "type": "api_key",
                        "key": search_key
                    },
                    "embedding_dependency": {
                        "type": "deployment_name",
                        "deployment_name": "text-embedding-3-large"
                    }
                }
            }]
        }
    )

    # OpenAI 응답 텍스트
    response_text = completion.choices[0].message.content
    print(f"OpenAI 응답: {response_text}")

    # Azure Speech로 TTS
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    speech_config.speech_synthesis_voice_name = voice_name
    speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config)
    result = speech_synthesizer.speak_text_async(response_text).get()

    return {
        "text_response": response_text,
        "openai_completion": completion.to_json()
    }

# 사용 예시
if __name__ == "__main__":
    result = azure_openai_with_speech("안녕하세요. 도움이 필요합니다.")
    print("함수 실행 결과:")
    print(json.dumps(result, indent=2, ensure_ascii=False))